Phase 1 - Setup

In [1]:
%%bash
pip install -q "qdrant-client[fastembed]>=1.14.2" sentence-transformers rapidfuzz pyarrow pandas tqdm

if [ ! -f /kaggle/working/qdrant_bin/qdrant ]; then
  mkdir -p /kaggle/working/qdrant_bin /kaggle/working/qdrant_storage
  wget -q https://github.com/qdrant/qdrant/releases/latest/download/qdrant-x86_64-unknown-linux-musl.tar.gz -O /tmp/qdrant.tar.gz
  tar -xzf /tmp/qdrant.tar.gz -C /kaggle/working/qdrant_bin
  chmod +x /kaggle/working/qdrant_bin/qdrant
fi

In [2]:
import os, subprocess, time, urllib.request, json

os.makedirs("/kaggle/working/qdrant_storage", exist_ok=True)
os.makedirs("/kaggle/working/artifacts", exist_ok=True)

qdrant_cfg = """
storage:
  storage_path: /kaggle/working/qdrant_storage
service:
  http_port: 6333
  grpc_port: 6334
telemetry_disabled: true
"""
with open("/kaggle/working/qdrant_config.yaml", "w") as f:
    f.write(qdrant_cfg)

qdrant_proc = subprocess.Popen(
    ["/kaggle/working/qdrant_bin/qdrant", "--config-path", "/kaggle/working/qdrant_config.yaml"],
    stdout=open("/tmp/qdrant.log", "w"),
    stderr=subprocess.STDOUT,
)

for _ in range(30):
    try:
        with urllib.request.urlopen("http://localhost:6333/healthz", timeout=1) as r:
            if r.status == 200:
                print("Qdrant server is UP on http://localhost:6333")
                break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("Qdrant failed to start. Check /tmp/qdrant.log")

Qdrant server is UP on http://localhost:6333


Phase 2 - Data loading

In [3]:
import glob, re
import numpy as np
import pandas as pd

parquet_candidates = glob.glob("/kaggle/input/**/recipes.parquet", recursive=True)
assert parquet_candidates, "Could not find recipes.parquet"
RECIPES_PATH = parquet_candidates[0]
print("Loading:", RECIPES_PATH)

cols = [
    "RecipeId", "Name", "CookTime", "PrepTime", "TotalTime",
    "RecipeCategory", "Keywords", "RecipeIngredientQuantities",
    "RecipeIngredientParts", "AggregatedRating", "ReviewCount",
    "RecipeInstructions", "Images"
]
df_raw = pd.read_parquet(RECIPES_PATH, columns=cols)
print(f"Raw rows: {len(df_raw):,}")

ISO_RE = re.compile(r"^P(?:(?P<days>\d+)D)?(?:T(?:(?P<hours>\d+)H)?(?:(?P<mins>\d+)M)?(?:(?P<secs>\d+)S)?)?$")

Loading: /kaggle/input/datasets/irkaal/foodcom-recipes-and-reviews/recipes.parquet
Raw rows: 522,517


In [4]:
df_raw[0:5]

,RecipeId,Name,CookTime,PrepTime,TotalTime,RecipeCategory,Keywords,RecipeIngredientQuantities,RecipeIngredientParts,AggregatedRating,ReviewCount,RecipeInstructions,Images
0,38.0,Low-Fat Berry Blue Frozen Dessert,PT24H,PT45M,PT24H45M,Frozen Desserts,"[Dessert, Low Protein, Low Cholesterol, Health...","[4, 1⁄4, 1, 1]","[blueberries, granulated sugar, vanilla yogurt...",4.5,4.0,"[Toss 2 cups berries with sugar., Let stand fo...",[https://img.sndimg.com/food/image/upload/w_55...
1,39.0,Biryani,PT25M,PT4H,PT4H25M,Chicken Breast,"[Chicken Thigh & Leg, Chicken, Poultry, Meat, ...","[1, 4, 2, 2, 8, 1⁄4, 8, 1⁄2, 1, 1, 1⁄4, 1⁄4, 1...","[saffron, milk, hot green chili peppers, onion...",3.0,1.0,[Soak saffron in warm milk for 5 minutes and p...,[https://img.sndimg.com/food/image/upload/w_55...
2,40.0,Best Lemonade,PT5M,PT30M,PT35M,Beverages,"[Low Protein, Low Cholesterol, Healthy, Summer...","[1 1⁄2, 1, None, 1 1⁄2, None, 3⁄4]","[sugar, lemons, rind of, lemon, zest of, fresh...",4.5,10.0,"[Into a 1 quart Jar with tight fitting lid, pu...",[https://img.sndimg.com/food/image/upload/w_55...
3,41.0,Carina's Tofu-Vegetable Kebabs,PT20M,PT24H,PT24H20M,Soy/Tofu,"[Beans, Vegetable, Low Cholesterol, Weeknight,...","[12, 1, 2, 1, 10, 1, 3, 2, 2, 2, 1, 2, 1⁄2, 1⁄...","[extra firm tofu, eggplant, zucchini, mushroom...",4.5,2.0,"[Drain the tofu, carefully squeezing out exces...",[https://img.sndimg.com/food/image/upload/w_55...
4,42.0,Cabbage Soup,PT30M,PT20M,PT50M,Vegetable,"[Low Protein, Vegan, Low Cholesterol, Healthy,...","[46, 4, 1, 2, 1]","[plain tomato juice, cabbage, onion, carrots, ...",4.5,11.0,"[Mix everything together and bring to a boil.,...",[https://img.sndimg.com/food/image/upload/w_55...


In [5]:
def parse_iso_minutes(val) -> int | None:
    if val is None or (isinstance(val, float) and np.isnan(val)):
        return None
    s = str(val).strip()
    m = ISO_RE.match(s)
    if not m:
        return None
    days = int(m.group("days") or 0)
    hours = int(m.group("hours") or 0)
    mins = int(m.group("mins") or 0)
    secs = int(m.group("secs") or 0)
    total = days * 1440 + hours * 60 + mins + (1 if secs >= 30 else 0)
    return total if total > 0 else None

def to_py_list(val) -> list[str]:
    if val is None:
        return []
    if isinstance(val, (list, tuple, np.ndarray)):
        return [str(x).strip() for x in val if x is not None and str(x).strip()]
    return []

In [6]:
df = df_raw.copy()
df["total_minutes"] = df["TotalTime"].apply(parse_iso_minutes)
df["ingredients_raw_parts"] = df["RecipeIngredientParts"].apply(to_py_list)
df["quantities_raw"] = df["RecipeIngredientQuantities"].apply(to_py_list)
df["keywords_list"] = df["Keywords"].apply(to_py_list)
df["steps_list"] = df["RecipeInstructions"].apply(to_py_list)
df["images_list"] = df["Images"].apply(to_py_list)

In [7]:
# Filter valid recipes: 1..480 minutes, >= 2 ingredients, non-empty title & instructions
df = df[
    df["Name"].notna()
    & (df["total_minutes"].between(1, 480))
    & (df["ingredients_raw_parts"].apply(len).between(2, 30))
    & (df["steps_list"].apply(len) >= 1)
].copy()

df["ReviewCount"] = df["ReviewCount"].fillna(0.0)
df["AggregatedRating"] = df["AggregatedRating"].fillna(0.0)

# Prioritise well-reviewed recipes, take 60,000
TARGET_N = 60_000
df = df.sort_values(["ReviewCount", "AggregatedRating", "RecipeId"], ascending=[False, False, True])
df = df.head(TARGET_N).reset_index(drop=True)
print(f"Filtered dataset size: {len(df):,} recipes")

Filtered dataset size: 60,000 recipes


In [8]:
def combine_qty_ing(qtys, parts):
    out = []
    for i, p in enumerate(parts):
        q = qtys[i] if i < len(qtys) and qtys[i] not in ("NA", "nan", "") else ""
        out.append(f"{q} {p}".strip() if q else p)
    return out

df["ingredients_raw"] = [combine_qty_ing(q, p) for q, p in zip(df["quantities_raw"], df["ingredients_raw_parts"])]
df["source_url"] = df["RecipeId"].astype(int).apply(lambda rid: f"https://www.food.com/recipe/{rid}")
df["image_url"] = df["images_list"].apply(lambda xs: xs[0] if xs else "")

Phase 3 - Ingredient Normalisation 

In [9]:
import re
from collections import Counter
from rapidfuzz import process, fuzz

PREP_WORDS = {
    "fresh", "freshly", "chopped", "finely", "roughly", "coarsely", "diced", "minced",
    "sliced", "thinly", "thickly", "grated", "shredded", "crushed", "ground", "melted",
    "softened", "room", "temperature", "cold", "warm", "hot", "boiling", "chilled",
    "frozen", "thawed", "canned", "drained", "rinsed", "packed", "lightly", "firmly",
    "divided", "optional", "to", "taste", "for", "garnish", "serving", "peeled",
    "seeded", "deveined", "trimmed", "halved", "quartered", "cubed", "julienned",
    "beaten", "whisked", "sifted", "unsalted", "salted", "sweetened", "unsweetened",
    "low", "fat", "nonfat", "reduced", "sodium", "organic", "raw", "cooked", "roasted",
    "toasted", "dried", "dry", "lean", "extra", "virgin", "pure", "plain", "whole",
    "large", "medium", "small", "tiny", "jumbo", "ripe", "unripe", "boneless",
    "skinless", "skin", "on", "bone", "in", "uncooked", "prepared", "instant",
}

UNIT_WORDS = {
    "cup", "cups", "tbsp", "tablespoon", "tablespoons", "tsp", "teaspoon", "teaspoons",
    "oz", "ounce", "ounces", "lb", "lbs", "pound", "pounds", "g", "gram", "grams",
    "kg", "ml", "l", "liter", "liters", "pinch", "dash", "clove", "cloves", "can",
    "cans", "jar", "jars", "package", "packages", "pkg", "pkt", "packet", "packets",
    "bag", "bags", "bunch", "bunches", "sprig", "sprigs", "stalk", "stalks", "head",
    "heads", "slice", "slices", "piece", "pieces", "strip", "strips", "drop", "drops",
    "stick", "sticks", "bottle", "bottles", "box", "boxes", "inch", "inches", "link", "links",
}

SYNONYMS = {
    # Produce & aromatics
    "baby spinach": "spinach", "spinach leaf": "spinach", "spinach leaves": "spinach",
    "fresh spinach": "spinach", "frozen spinach": "spinach",
    "yellow onion": "onion", "white onion": "onion", "red onion": "onion", "sweet onion": "onion",
    "vidalia onion": "onion", "spanish onion": "onion", "onions": "onion",
    "green onion": "scallion", "green onions": "scallion", "spring onion": "scallion", "scallions": "scallion",
    "garlic clove": "garlic", "garlic cloves": "garlic", "garlic powder": "garlic powder",
    "roma tomato": "tomato", "plum tomato": "tomato", "italian plum tomato": "tomato",
    "mexican style tomato": "tomato", "cherry tomato": "tomato", "grape tomato": "tomato",
    "tomatoes": "tomato", "canned tomato": "tomato", "diced tomato": "tomato", "crushed tomato": "tomato",
    "bell pepper": "bell pepper", "red bell pepper": "bell pepper", "green bell pepper": "bell pepper",
    "yellow bell pepper": "bell pepper", "sweet pepper": "bell pepper",
    "russet potato": "potato", "russet baking potato": "potato", "yukon gold potato": "potato",
    "red potato": "potato", "potatoes": "potato",
    "sweet potatoes": "sweet potato", "yam": "sweet potato",
    "green seedless grape": "grape", "red seedless grape": "grape",
    "cilantro leaf": "cilantro", "coriander leaf": "cilantro", "fresh cilantro": "cilantro",
    "flat leaf parsley": "parsley", "italian parsley": "parsley",
    "lemon juice": "lemon juice", "fresh lemon juice": "lemon juice",
    "lime juice": "lime juice", "fresh lime juice": "lime juice",
    # Grains, flours & gluten sources
    "all purpose flour": "wheat flour", "all-purpose flour": "wheat flour", "plain flour": "wheat flour",
    "white flour": "wheat flour", "unbleached flour": "wheat flour", "bread flour": "wheat flour",
    "cake flour": "wheat flour", "pastry flour": "wheat flour", "self rising flour": "wheat flour",
    "self-rising flour": "wheat flour", "whole wheat flour": "whole wheat flour", "flour": "wheat flour",
    "cornstarch": "cornstarch", "corn starch": "cornstarch", "cornflour": "cornstarch",
    "cornmeal": "cornmeal", "polenta": "cornmeal",
    "almond flour": "almond flour", "almond meal": "almond flour",
    "coconut flour": "coconut flour", "rice flour": "rice flour", "tapioca flour": "tapioca starch",
    "oat": "oats", "rolled oat": "oats", "rolled oats": "oats", "old fashioned oats": "oats", "quick oats": "oats",
    "white rice": "rice", "long grain rice": "rice", "basmati rice": "rice", "jasmine rice": "rice",
    "brown rice": "rice", "arborio rice": "rice", "cooked rice": "rice",
    "spaghetti": "pasta", "penne": "pasta", "fettuccine": "pasta", "linguine": "pasta",
    "macaroni": "pasta", "rigatoni": "pasta", "rotini": "pasta", "fusilli": "pasta", "egg noodle": "egg noodles",
    "bread crumb": "breadcrumbs", "bread crumbs": "breadcrumbs", "panko": "breadcrumbs", "panko breadcrumbs": "breadcrumbs",
    # Pantry, alcohol & sauces
    "extra virgin olive oil": "olive oil", "virgin olive oil": "olive oil",
    "canola oil": "vegetable oil", "sunflower oil": "vegetable oil", "corn oil": "vegetable oil",
    "kosher salt": "salt", "sea salt": "salt", "table salt": "salt", "canning salt": "salt", "hickory smoke salt": "salt",
    "black pepper": "black pepper", "ground black pepper": "black pepper", "pepper": "black pepper",
    "granulated sugar": "sugar", "white sugar": "sugar", "caster sugar": "sugar",
    "light brown sugar": "brown sugar", "dark brown sugar": "brown sugar",
    "powdered sugar": "powdered sugar", "confectioners sugar": "powdered sugar", "icing sugar": "powdered sugar",
    "tamari": "tamari", "gluten free soy sauce": "tamari", "gluten-free soy sauce": "tamari",
    "light soy sauce": "soy sauce", "dark soy sauce": "soy sauce", "low sodium soy sauce": "soy sauce",
    "chicken broth": "chicken stock", "chicken stock": "chicken stock",
    "beef broth": "beef stock", "beef stock": "beef stock",
    "vegetable broth": "vegetable stock", "vegetable stock": "vegetable stock",
    "dark beer": "beer", "light beer": "beer",
    # Dairy & eggs & vegan alternatives
    "eggs": "egg", "large egg": "egg", "egg yolk": "egg", "egg white": "egg", "egg whites": "egg",
    "whole milk": "milk", "skim milk": "milk", "2% milk": "milk", "buttermilk": "buttermilk",
    "chocolate milk": "chocolate milk",
    "heavy cream": "heavy cream", "whipping cream": "heavy cream", "heavy whipping cream": "heavy cream",
    "half and half": "cream", "half-and-half": "cream", "sour cream": "sour cream",
    "cream cheese": "cream cheese", "cheddar cheese": "cheddar", "sharp cheddar": "cheddar",
    "parmesan cheese": "parmesan", "parmigiano reggiano": "parmesan", "mozzarella cheese": "mozzarella",
    "feta cheese": "feta", "goat cheese": "goat cheese", "ricotta cheese": "ricotta",
    "unsalted butter": "butter", "salted butter": "butter",
    "almond milk": "almond milk", "soy milk": "soy milk", "oat milk": "oat milk", "coconut milk": "coconut milk",
    # Proteins
    "chicken breast": "chicken", "chicken breasts": "chicken", "chicken thigh": "chicken",
    "chicken thighs": "chicken", "chicken tender": "chicken", "rotisserie chicken": "chicken", "chicken meat": "chicken",
    "ground beef": "beef", "beef chuck": "beef", "steak": "beef", "stew beef": "beef",
    "round steak": "beef", "beef sirloin": "beef",
    "pork chop": "pork", "pork tenderloin": "pork", "ground pork": "pork",
    "sweet italian sausage": "sausage", "italian sausage": "sausage",
    "turkey breast": "turkey", "ground turkey": "turkey",
    "garbanzo bean": "chickpeas", "garbanzo beans": "chickpeas", "chickpea": "chickpeas",
    "black beans": "black bean", "kidney beans": "kidney bean", "cannellini beans": "white bean",
    "firm tofu": "tofu", "extra firm tofu": "tofu", "silken tofu": "tofu",
    # Nuts & seeds
    "peanut": "peanuts", "peanut butter": "peanut butter", "smooth peanut butter": "peanut butter",
    "almond": "almonds", "sliced almonds": "almonds", "slivered almonds": "almonds",
    "walnut": "walnuts", "pecan": "pecans", "cashew": "cashews", "pistachio": "pistachios",
    "hazelnut": "hazelnuts", "pine nut": "pine nuts", "pine nuts": "pine nuts",
}

In [10]:
def simple_singularise(phrase: str) -> str:
    words = phrase.split()
    if not words:
        return phrase
    last = words[-1]
    keep = {"oats", "chickpeas", "lentils", "almonds", "walnuts", "pecans", "cashews",
            "peanuts", "pistachios", "hazelnuts", "breadcrumbs", "noodles", "egg noodles",
            "berries", "molasses", "couscous", "hummus", "asparagus"}
    if phrase in keep or last in keep:
        return phrase
    if len(last) > 4 and last.endswith("ies"):
        words[-1] = last[:-3] + "y"
    elif len(last) > 4 and last.endswith("oes"):
        words[-1] = last[:-2]
    elif len(last) > 3 and last.endswith("s") and not last.endswith(("ss", "us", "is")):
        words[-1] = last[:-1]
    return " ".join(words)

CLEAN_PARENS_RE = re.compile(r"\([^)]*\)")
NON_ALPHA_RE = re.compile(r"[^a-z0-9\s\-]")
QTY_PREFIX_RE = re.compile(r"^[\d\s/.\-]+")

def clean_raw_line(raw: str) -> str:
    s = raw.lower().strip()
    s = CLEAN_PARENS_RE.sub(" ", s)
    s = s.split(",")[0]
    s = QTY_PREFIX_RE.sub("", s)
    s = s.replace("-", " ")
    s = NON_ALPHA_RE.sub(" ", s)
    tokens = [t for t in s.split() if t not in UNIT_WORDS and t not in PREP_WORDS and not t.isdigit()]
    return " ".join(tokens).strip()

raw_counter = Counter()
for parts in df["ingredients_raw_parts"]:
    for p in parts:
        c = clean_raw_line(p)
        if c:
            c = SYNONYMS.get(c, simple_singularise(c))
            c = SYNONYMS.get(c, c)
            if c:
                raw_counter[c] += 1

TOP_CANONICAL = {ing for ing, _ in raw_counter.most_common(1500)}
TOP_CANONICAL.update(SYNONYMS.values())
KNOWN_VOCAB = sorted(TOP_CANONICAL)

fuzzy_cache: dict[str, str] = {}
unmatched_counter = Counter()

In [11]:

def normalise_ingredient(raw: str) -> tuple[str, bool]:
    """Returns (canonical_name, is_known)."""
    c = clean_raw_line(raw)
    if not c:
        return ("", False)
    if c in SYNONYMS:
        return (SYNONYMS[c], True)
    cs = simple_singularise(c)
    if cs in SYNONYMS:
        return (SYNONYMS[cs], True)
    if cs in TOP_CANONICAL:
        return (cs, True)
    if c in TOP_CANONICAL:
        return (c, True)
    if cs in fuzzy_cache:
        m = fuzzy_cache[cs]
        if m == "__UNMATCHED__":
            unmatched_counter[cs] += 1
            return (cs, False)
        return (m, True)
    if len(cs) >= 5:
        hit = process.extractOne(cs, KNOWN_VOCAB, scorer=fuzz.WRatio, score_cutoff=92)
        if hit:
            fuzzy_cache[cs] = hit[0]
            return (hit[0], True)
    fuzzy_cache[cs] = "__UNMATCHED__"
    unmatched_counter[cs] += 1
    return (cs, False)

In [12]:
canon_lists = []
unknown_flags = []
for parts in df["ingredients_raw_parts"]:
    c_set = []
    has_unknown = False
    for p in parts:
        c, known = normalise_ingredient(p)
        if c:
            if c not in c_set:
                c_set.append(c)
            if not known:
                has_unknown = True
    canon_lists.append(c_set)
    unknown_flags.append(has_unknown)

df["ingredients"] = canon_lists
df["ingredient_count"] = df["ingredients"].apply(len)
df["has_unknown_ingredient"] = unknown_flags

print(f"Unique canonical vocab size: {len(KNOWN_VOCAB)}")
print(f"Recipes with >=1 unrecognised ingredient: {df['has_unknown_ingredient'].sum():,} / {len(df):,}")
print("Top 15 unmatched tokens:", unmatched_counter.most_common(15))

Unique canonical vocab size: 1510
Recipes with >=1 unrecognised ingredient: 2,471 / 60,000
Top 15 unmatched tokens: [('chocolate soymilk', 6), ('chunk tuna', 6), ('wheat pasta', 6), ('jell o chocolate pudding and pie filling', 6), ('white vermouth', 6), ('lavender blossom', 6), ('country ham', 6), ('cos lettuce', 6), ('green asparagus', 6), ('greek feta cheese', 6), ('clementine', 6), ('chocolate covered coffee bean', 6), ('sugar free vanilla ice cream', 6), ('golden rum', 6), ('chinese mustard', 6)]


Phase 4 - Conservative Dietary Tag & Allergen Derivation

In [13]:
GF_EXCEPTIONS = {
    "tamari", "gluten free soy sauce", "gluten-free soy sauce", "gluten free flour",
    "gluten-free flour", "almond flour", "coconut flour", "rice flour", "chickpea flour",
    "tapioca starch", "tapioca flour", "cornstarch", "potato starch", "arrowroot",
    "buckwheat", "quinoa", "rice", "cornmeal", "polenta", "certified gluten free oats",
    "gluten free bread", "gluten free pasta", "gluten-free pasta", "rice noodles", "rice paper",
    "apple cider vinegar", "white vinegar", "balsamic vinegar", "wine vinegar", "rice vinegar",
}
GLUTEN_VIOLATIONS = {
    "wheat", "wheat flour", "whole wheat", "whole wheat flour", "flour", "barley", "rye",
    "spelt", "kamut", "farro", "semolina", "durum", "couscous", "bulgur", "seitan",
    "pasta", "spaghetti", "macaroni", "noodles", "egg noodles", "ramen", "udon",
    "bread", "breadcrumbs", "panko", "croutons", "tortilla", "flour tortilla", "pita",
    "bagel", "cracker", "crackers", "biscuit", "pastry", "puff pastry", "phyllo",
    "soy sauce", "teriyaki", "teriyaki sauce", "hoisin", "beer", "ale", "lager", "stout",
    "malt", "malt vinegar", "graham cracker", "matzo", "orzo",
}
GLUTEN_AMBIGUOUS = {
    "oats", "oatmeal", "oat flour", "chicken stock", "beef stock", "vegetable stock",
    "broth", "bouillon", "worcestershire", "worcestershire sauce", "seasoning",
    "taco seasoning", "curry paste", "miso", "gravy", "baking mix", "cake mix",
    "soup mix", "dressing", "barbecue sauce", "bbq sauce", "marinade",
}

HALAL_EXCEPTIONS = {
    "apple cider vinegar", "wine vinegar", "red wine vinegar", "white wine vinegar",
    "balsamic vinegar", "rice vinegar", "sherry vinegar", "ginger ale", "root beer",
    "turkey bacon", "beef bacon", "halal gelatin", "agar", "agar agar", "pectin",
    "beef sausage", "chicken sausage", "turkey sausage", "lamb sausage", "vegan sausage",
    "chicken stock", "chicken broth", "beef stock", "beef broth",
    "vegetable stock", "vegetable broth",
    "coconut rum extract", "vanilla", "imitation vanilla",
}
HALAL_VIOLATIONS = {
    "pork", "bacon", "ham", "prosciutto", "pancetta", "lard", "pepperoni", "chorizo",
    "pork chop", "pork belly", "pork loin", "pork shoulder", "spareribs", "gammon",
    "wine", "red wine", "white wine", "cooking wine", "marsala", "sherry", "champagne",
    "prosecco", "beer", "ale", "stout", "lager", "rum", "bourbon", "whiskey", "whisky",
    "vodka", "gin", "brandy", "cognac", "tequila", "sake", "mirin", "liqueur",
    "amaretto", "kahlua", "grand marnier", "vermouth", "port", "hard cider", "gelatin",
}
HALAL_AMBIGUOUS = {
    "sausage", "salami", "bologna", "hot dog", "hot dogs", "meatballs", "stock",
    "broth", "bouillon", "vanilla extract", "rum extract", "almond extract",
    "marshmallows", "marshmallow", "worcestershire", "worcestershire sauce",
}

VEGAN_EXCEPTIONS = {
    "coconut milk", "almond milk", "soy milk", "oat milk", "rice milk", "cashew milk",
    "peanut butter", "almond butter", "cashew butter", "sunflower butter", "apple butter",
    "cocoa butter", "coconut butter", "nutritional yeast", "vegan cheese", "vegan butter",
    "tofu", "tempeh", "seitan", "agave", "maple syrup", "vegetable stock", "vegetable broth",
    "coconut oil", "olive oil", "vegetable oil", "canola oil", "sesame oil",
    "coconut cream", "vegan mayonnaise", "agar",
    "gluten-free pasta", "gluten free pasta", "rice noodles",
}
ANIMAL_MEAT_FISH = {
    "beef", "chicken", "pork", "turkey", "lamb", "veal", "duck", "goose", "venison",
    "bacon", "ham", "prosciutto", "pancetta", "sausage", "salami", "pepperoni", "chorizo",
    "fish", "salmon", "tuna", "cod", "tilapia", "trout", "halibut", "anchovy", "anchovies",
    "shrimp", "prawn", "crab", "lobster", "scallop", "clam", "mussel", "oyster", "squid",
    "chicken stock", "beef stock", "fish sauce", "oyster sauce", "worcestershire",
    "worcestershire sauce", "lard", "tallow", "gelatin",
}
DAIRY_EGG_HONEY = {
    "milk", "chocolate milk", "buttermilk", "cream", "heavy cream", "whipping cream",
    "sour cream", "half and half", "butter", "ghee", "cheese", "cheddar", "parmesan",
    "mozzarella", "feta", "goat cheese", "ricotta", "cream cheese", "brie", "swiss cheese",
    "yogurt", "greek yogurt", "whey", "casein", "condensed milk", "evaporated milk",
    "egg", "egg noodles", "mayonnaise", "mayo", "honey", "milk chocolate", "white chocolate",
}
VEGAN_AMBIGUOUS = {
    "margarine", "chocolate", "chocolate chips", "pesto", "stock", "broth", "bouillon",
    "curry paste", "kimchi", "bread", "fresh pasta", "egg pasta", "ravioli", "tortellini", "gnocchi",
}

NUT_EXCEPTIONS = {
    "coconut", "coconut milk", "coconut oil", "coconut flour", "coconut cream",
    "nutmeg", "water chestnut", "water chestnuts", "butternut squash",
}
NUT_VIOLATIONS = {
    "peanut", "peanuts", "peanut butter", "peanut oil", "almond", "almonds",
    "almond flour", "almond meal", "almond milk", "almond butter", "almond extract",
    "walnut", "walnuts", "pecan", "pecans", "cashew", "cashews", "cashew milk",
    "cashew butter", "pistachio", "pistachios", "hazelnut", "hazelnuts", "filbert",
    "macadamia", "brazil nut", "pine nut", "pine nuts", "pesto", "marzipan",
    "nutella", "praline", "mixed nuts", "tree nuts",
}
NUT_AMBIGUOUS = {
    "nut oil", "satay", "granola", "muesli", "trail mix",
}

In [14]:
def _match_set(token: str, vocab: set[str]) -> bool:
    if token in vocab:
        return True
    words = set(token.split())
    for term in vocab:
        if " " in term:
            if term in token:
                return True
        elif term in words:
            return True
    return False

def evaluate_diets(ingredients: list[str], has_unknown: bool) -> tuple[list[str], dict[str, str], list[str]]:
    status = {
        "gluten_free": "derived",
        "halal_friendly": "derived",
        "vegan": "derived",
        "nut_free": "derived",
    }
    allergens = set()

    for raw_tok in ingredients:
        tok = raw_tok.lower().strip()
        if not tok:
            continue

        # 1. Gluten check
        if not _match_set(tok, GF_EXCEPTIONS):
            if _match_set(tok, GLUTEN_VIOLATIONS):
                status["gluten_free"] = "violated"
                allergens.add("gluten")
            elif _match_set(tok, GLUTEN_AMBIGUOUS) and status["gluten_free"] != "violated":
                status["gluten_free"] = "unknown"

        # 2. Halal check
        if not _match_set(tok, HALAL_EXCEPTIONS):
            if _match_set(tok, HALAL_VIOLATIONS):
                status["halal_friendly"] = "violated"
                if any(k in tok for k in ("pork", "bacon", "ham", "lard", "prosciutto", "pancetta")):
                    allergens.add("pork")
                else:
                    allergens.add("alcohol_or_gelatin")
            elif _match_set(tok, HALAL_AMBIGUOUS) and status["halal_friendly"] != "violated":
                status["halal_friendly"] = "unknown"

        # 3. Vegan check
        if not _match_set(tok, VEGAN_EXCEPTIONS):
            if _match_set(tok, ANIMAL_MEAT_FISH) or _match_set(tok, DAIRY_EGG_HONEY):
                status["vegan"] = "violated"
                if _match_set(tok, DAIRY_EGG_HONEY):
                    if any(w in tok for w in ("milk", "cream", "butter", "cheese", "cheddar", "parmesan", "mozzarella", "yogurt", "ghee", "whey")):
                        allergens.add("dairy")
                    if "egg" in tok.split() or "mayo" in tok:
                        allergens.add("egg")
                if any(w in tok for w in ("fish", "salmon", "tuna", "cod", "anchovy")):
                    allergens.add("fish")
                if any(w in tok for w in ("shrimp", "crab", "lobster", "prawn", "scallop")):
                    allergens.add("shellfish")
            elif _match_set(tok, VEGAN_AMBIGUOUS) and status["vegan"] != "violated":
                status["vegan"] = "unknown"

        # 4. Nut check
        if not _match_set(tok, NUT_EXCEPTIONS):
            if _match_set(tok, NUT_VIOLATIONS):
                status["nut_free"] = "violated"
                if "peanut" in tok:
                    allergens.add("peanut")
                else:
                    allergens.add("tree_nut")
            elif _match_set(tok, NUT_AMBIGUOUS) and status["nut_free"] != "violated":
                status["nut_free"] = "unknown"

        if any(w in tok for w in ("soy", "tofu", "tamari", "edamame", "miso")):
            allergens.add("soy")
        if "sesame" in tok or "tahini" in tok:
            allergens.add("sesame")

    if has_unknown:
        for k in status:
            if status[k] == "derived":
                status[k] = "unknown"

    diet_tags = [k for k, v in status.items() if v == "derived"]
    return diet_tags, status, sorted(allergens)


In [15]:
TEST_CASES = [
    ("Spinach egg fried rice w/ soy sauce", ["rice", "egg", "spinach", "soy sauce"], False, ["halal_friendly", "nut_free"], ["gluten_free", "vegan"]),
    ("Spinach egg fried rice w/ tamari", ["rice", "egg", "spinach", "tamari"], False, ["gluten_free", "halal_friendly", "nut_free"], ["vegan"]),
    ("Chickpea tomato curry w/ coconut milk", ["chickpeas", "tomato", "onion", "garlic", "coconut milk"], False, ["gluten_free", "halal_friendly", "vegan", "nut_free"], []),
    ("Pasta with basil pesto", ["pasta", "pesto", "olive oil"], False, ["halal_friendly"], ["gluten_free", "nut_free", "vegan"]),
    ("Gluten-free pasta with tomato & olive oil", ["gluten-free pasta", "tomato", "olive oil", "garlic"], False, ["gluten_free", "halal_friendly", "vegan", "nut_free"], []),
    ("Almond flour banana pancakes", ["almond flour", "banana", "egg", "milk"], False, ["gluten_free", "halal_friendly"], ["nut_free", "vegan"]),
    ("Oat banana smoothie (unverified oats)", ["oats", "banana", "almond milk"], False, ["halal_friendly", "vegan"], ["gluten_free", "nut_free"]),
    ("Certified GF oats with soy milk", ["certified gluten free oats", "soy milk", "banana"], False, ["gluten_free", "halal_friendly", "vegan", "nut_free"], []),
    ("Caesar salad w/ Worcestershire sauce", ["romaine", "parmesan", "worcestershire sauce", "olive oil"], False, ["nut_free"], ["gluten_free", "halal_friendly", "vegan"]),
    ("Chicken soup w/ generic chicken stock", ["chicken", "carrot", "celery", "chicken stock"], False, ["halal_friendly", "nut_free"], ["gluten_free", "vegan"]),
    ("Vegetable soup w/ vegetable stock", ["carrot", "celery", "potato", "vegetable stock"], False, ["halal_friendly", "vegan", "nut_free"], ["gluten_free"]),
    ("Beef bourguignon w/ red wine & bacon", ["beef", "red wine", "bacon", "carrot"], False, ["nut_free"], ["halal_friendly", "vegan"]),
    ("Grilled lamb with apple cider vinegar", ["lamb", "olive oil", "garlic", "apple cider vinegar"], False, ["gluten_free", "halal_friendly", "nut_free"], ["vegan"]),
    ("Honey glazed carrots w/ butter", ["carrot", "honey", "butter"], False, ["gluten_free", "halal_friendly", "nut_free"], ["vegan"]),
    ("Maple glazed carrots w/ olive oil", ["carrot", "maple syrup", "olive oil"], False, ["gluten_free", "halal_friendly", "vegan", "nut_free"], []),
    ("Peanut noodle stir fry", ["rice noodles", "peanut butter", "tamari", "broccoli"], False, ["gluten_free", "halal_friendly", "vegan"], ["nut_free"]),
    ("Sesame tofu stir fry w/ mirin", ["tofu", "sesame oil", "broccoli", "mirin"], False, ["gluten_free", "vegan", "nut_free"], ["halal_friendly"]),
    ("Turkey bacon & egg breakfast", ["turkey bacon", "egg", "spinach"], False, ["gluten_free", "halal_friendly", "nut_free"], ["vegan"]),
    ("Pork sausage & potato hash", ["sausage", "potato", "onion"], False, ["gluten_free", "nut_free"], ["halal_friendly", "vegan"]),
    ("Beef sausage & potato hash", ["beef sausage", "potato", "onion"], False, ["gluten_free", "halal_friendly", "nut_free"], ["vegan"]),
    ("Butternut squash soup w/ nutmeg", ["butternut squash", "nutmeg", "coconut cream", "onion"], False, ["gluten_free", "halal_friendly", "vegan", "nut_free"], []),
    ("Vegan mayo potato salad", ["potato", "vegan mayonnaise", "scallion", "mustard"], False, ["gluten_free", "halal_friendly", "vegan", "nut_free"], []),
    ("Regular mayo potato salad", ["potato", "mayonnaise", "egg", "scallion"], False, ["gluten_free", "halal_friendly", "nut_free"], ["vegan"]),
    ("Beer battered cod", ["cod", "wheat flour", "beer", "salt"], False, ["nut_free"], ["gluten_free", "halal_friendly", "vegan"]),
    ("Cornstarch fried tofu", ["tofu", "cornstarch", "vegetable oil", "salt"], False, ["gluten_free", "halal_friendly", "vegan", "nut_free"], []),
    ("Gelatin berry dessert", ["strawberry", "sugar", "gelatin", "water"], False, ["gluten_free", "nut_free"], ["halal_friendly", "vegan"]),
    ("Agar berry dessert", ["strawberry", "sugar", "agar", "water"], False, ["gluten_free", "halal_friendly", "vegan", "nut_free"], []),
    ("Walnut brownie", ["wheat flour", "cocoa powder", "butter", "egg", "walnuts"], False, ["halal_friendly"], ["gluten_free", "vegan", "nut_free"]),
    ("Unknown exotic powder blocks certification", ["rice", "tomato", "mystery_spice_xyz"], True, [], ["gluten_free", "halal_friendly", "vegan", "nut_free"]),
    ("Rice & black bean bowl w/ avocado", ["rice", "black bean", "avocado", "lime juice", "cilantro"], False, ["gluten_free", "halal_friendly", "vegan", "nut_free"], []),
]

for name, ings, unk, exp_safe, exp_not_safe in TEST_CASES:
    tags, conf, _ = evaluate_diets(ings, unk)
    for t in exp_safe:
        assert t in tags, f"FAIL [{name}]: expected '{t}' in {tags} (conf={conf})"
    for t in exp_not_safe:
        assert t not in tags, f"FAIL [{name}]: did NOT expect '{t}' in {tags} (conf={conf})"
print(f"All {len(TEST_CASES)}/30 dietary tag regression tests PASSED!")

results = [evaluate_diets(ings, unk) for ings, unk in zip(df["ingredients"], df["has_unknown_ingredient"])]
df["diet_tags"] = [r[0] for r in results]
df["diet_confidence"] = [r[1] for r in results]
df["allergens_present"] = [r[2] for r in results]

for tag in ["gluten_free", "halal_friendly", "vegan", "nut_free"]:
    cnt = df["diet_tags"].apply(lambda xs: tag in xs).sum()
    print(f"Certified '{tag}': {cnt:,} recipes ({cnt/len(df)*100:.1f}%)")

All 30/30 dietary tag regression tests PASSED!
Certified 'gluten_free': 29,184 recipes (48.6%)
Certified 'halal_friendly': 45,882 recipes (76.5%)
Certified 'vegan': 10,134 recipes (16.9%)
Certified 'nut_free': 52,270 recipes (87.1%)


Phase 5 - Computing dense & sparse vectors

In [16]:
import json, torch
from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer
from fastembed import SparseTextEmbedding
from qdrant_client import QdrantClient, models

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Embedding device:", device)

DENSE_MODEL_NAME = "BAAI/bge-small-en-v1.5"
SPARSE_MODEL_NAME = "Qdrant/bm25"
COLLECTION_NAME = "recipes"

dense_model = SentenceTransformer(DENSE_MODEL_NAME, device=device)
sparse_model = SparseTextEmbedding(model_name=SPARSE_MODEL_NAME)

# Build embedding text: title + canonical ingredient list (no noisy steps)
embed_texts = [
    f"{title}. Ingredients: {', '.join(ings)}"
    for title, ings in zip(df["Name"], df["ingredients"])
]

Embedding device: cuda


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

... (6 outputs left out)

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 18 files:   0%|          | 0/18 [00:00<?, ?it/s]

In [17]:
print("Computing dense embeddings on GPU...")
dense_vectors = dense_model.encode(
    embed_texts,
    batch_size=256,
    show_progress_bar=True,
    normalize_embeddings=True,
)

Computing dense embeddings on GPU...


Batches:   0%|          | 0/235 [00:00<?, ?it/s]

In [18]:
print("Computing BM25 sparse vectors...")
sparse_vectors = list(tqdm(sparse_model.embed(embed_texts, batch_size=256), total=len(embed_texts)))

Computing BM25 sparse vectors...


  0%|          | 0/60000 [00:00<?, ?it/s]

In [19]:
# Connect to local Qdrant binary
client = QdrantClient(url="http://localhost:6333", prefer_grpc=True, timeout=60)

if client.collection_exists(COLLECTION_NAME):
    client.delete_collection(COLLECTION_NAME)

client.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config={
        "dense": models.VectorParams(size=384, distance=models.Distance.COSINE),
    },
    sparse_vectors_config={
        "bm25": models.SparseVectorParams(modifier=models.Modifier.IDF),
    },
)

# Payload indexes for fast filtered HNSW
for field in ["diet_tags", "allergens_present", "ingredients", "meal_type"]:
    client.create_payload_index(COLLECTION_NAME, field_name=field, field_schema=models.PayloadSchemaType.KEYWORD)
client.create_payload_index(COLLECTION_NAME, field_name="total_minutes", field_schema=models.PayloadSchemaType.INTEGER)

BATCH_SIZE = 500
for start in tqdm(range(0, len(df), BATCH_SIZE), desc="Upserting to Qdrant"):
    end = min(start + BATCH_SIZE, len(df))
    points = []
    for idx in range(start, end):
        row = df.iloc[idx]
        sp = sparse_vectors[idx]
        payload = {
            "id": int(row["RecipeId"]),
            "title": str(row["Name"]),
            "source_url": str(row["source_url"]),
            "image_url": str(row["image_url"]),
            "ingredients_raw": list(row["ingredients_raw"]),
            "ingredients": list(row["ingredients"]),
            "ingredient_count": int(row["ingredient_count"]),
            "total_minutes": int(row["total_minutes"]),
            "diet_tags": list(row["diet_tags"]),
            "diet_confidence": dict(row["diet_confidence"]),
            "allergens_present": list(row["allergens_present"]),
            "meal_type": str(row["RecipeCategory"] or "Other"),
            "rating": float(row["AggregatedRating"]),
            "review_count": int(row["ReviewCount"]),
        }
        points.append(
            models.PointStruct(
                id=int(row["RecipeId"]),
                vector={
                    "dense": dense_vectors[idx].tolist(),
                    "bm25": models.SparseVector(
                        indices=sp.indices.tolist(),
                        values=sp.values.tolist(),
                    ),
                },
                payload=payload,
            )
        )
    client.upsert(collection_name=COLLECTION_NAME, points=points, wait=True)

info = client.get_collection(COLLECTION_NAME)
print(f"Collection '{COLLECTION_NAME}' indexed points: {info.points_count:,}")

Upserting to Qdrant:   0%|          | 0/120 [00:00<?, ?it/s]

Collection 'recipes' indexed points: 60,000


In [20]:
# Quick hybrid filtered smoke test
q_dense = dense_model.encode("rice egg spinach", normalize_embeddings=True).tolist()
q_sparse = next(sparse_model.query_embed("rice egg spinach"))
res = client.query_points(
    collection_name=COLLECTION_NAME,
    prefetch=[
        models.Prefetch(query=q_dense, using="dense", limit=20),
        models.Prefetch(
            query=models.SparseVector(indices=q_sparse.indices.tolist(), values=q_sparse.values.tolist()),
            using="bm25",
            limit=20,
        ),
    ],
    query=models.FusionQuery(fusion=models.Fusion.RRF),
    query_filter=models.Filter(
        must=[
            models.FieldCondition(key="diet_tags", match=models.MatchValue(value="gluten_free")),
            models.FieldCondition(key="total_minutes", range=models.Range(lte=20)),
        ]
    ),
    limit=3,
    with_payload=True,
)
print("Smoke test top 3 (gluten_free, <=20 min):")
for p in res.points:
    print(f" - {p.payload['title']} ({p.payload['total_minutes']} min) | ings={p.payload['ingredients']}")

# Save cleaned dataframe & taxonomy artifacts for Notebook 2
save_cols = [
    "RecipeId", "Name", "source_url", "image_url", "ingredients_raw",
    "ingredients", "ingredient_count", "total_minutes", "diet_tags",
    "diet_confidence", "allergens_present", "RecipeCategory",
    "AggregatedRating", "ReviewCount", "has_unknown_ingredient",
]
df[save_cols].to_parquet("/kaggle/working/artifacts/recipes_clean.parquet", index=False)

with open("/kaggle/working/artifacts/taxonomy.json", "w") as f:
    json.dump({"synonyms": SYNONYMS, "known_vocab": KNOWN_VOCAB}, f, indent=2)

# Cleanly flush & stop Qdrant so storage files aren't locked when saving Kaggle Output
client.close()
qdrant_proc.terminate()
qdrant_proc.wait(timeout=15)

Smoke test top 3 (gluten_free, <=20 min):
 - Scrambled Egg With Spinach and Feta on Toast (10 min) | ings=['egg', 'feta', 'olive oil']
 - Greek Chicken and Spinach (20 min) | ings=['chicken', 'lemon', 'oregano', 'garlic', 'red pepper flake', 'rice vinegar', 'olive oil', 'spinach', 'parsley', 'dill', 'feta']
 - Italian Spinach (12 min) | ings=['spinach', 'egg', 'garlic', 'parmesan']


0